In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
!pip install -q datasets
!pip install sentencepiece

In [ ]:
from datasets import load_dataset
dataset = load_dataset("nntsuzu/JESC")
print(dataset)

In [ ]:
print(dataset["train"][0])

In [ ]:
class CFG:
    SEED = 42


In [ ]:
dataset = dataset["train"].shuffle(seed=CFG.SEED)
small_dataset = dataset.select(range(50_000))
print(small_dataset)

In [ ]:
train_dataset = dataset.select(range(0, 2000000))
val_dataset   = dataset.select(range(2000000, 2801388))
#test_dataset  = dataset.select(range(47_500, 50_000))
print("train:", len(train_dataset))
print("validation:", len(val_dataset))
#print("test:", len(test_dataset))

In [ ]:
import os
import sentencepiece as spm

SP_DIR = "/content/gdrive/MyDrive/深層学習スクラッチ/deeplearning_implementation/Transformer/transformer_jesc/sentencepiece"
os.makedirs(SP_DIR, exist_ok=True)
EN_PREFIX = os.path.join(SP_DIR, "sp_en")
JA_PREFIX = os.path.join(SP_DIR, "sp_ja")
EN_MODEL = EN_PREFIX + ".model"
JA_MODEL = JA_PREFIX + ".model"


if os.path.exists(EN_MODEL) and os.path.exists(JA_MODEL):
    print("Saved SentencePiece models found.")
    print("Loading from Google Drive...")
else:
    print("SentencePiece models not found.")
    print("Training tokenizers...")
    with open("train_en.txt", "w", encoding="utf-8") as f:
        for sample in train_dataset:
            en = sample["translation"]["en"].strip()
            if en:
                f.write(en + "\n")

    with open("train_ja.txt", "w", encoding="utf-8") as f:
        for sample in train_dataset:
            ja = sample["translation"]["ja"].strip()
            if ja:
                f.write(ja + "\n")

    spm.SentencePieceTrainer.train(
        input="train_en.txt",
        model_prefix=EN_PREFIX,
        vocab_size=80000,
        model_type="bpe",
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<bos>",
        eos_piece="<eos>",
    )
    spm.SentencePieceTrainer.train(
        input="train_ja.txt",
        model_prefix=JA_PREFIX,
        vocab_size=80000,
        model_type="bpe",
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<bos>",
        eos_piece="<eos>",
    )

sp_en = spm.SentencePieceProcessor(model_file=EN_MODEL)
sp_ja = spm.SentencePieceProcessor(model_file=JA_MODEL)
print("English vocab:", sp_en.vocab_size())
print("Japanese vocab:", sp_ja.vocab_size())


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (B, T_q, d_k)
    K: (B, T_k, d_k)
    V: (B, T_k, d_k)
    """
    d_k = Q.size(-1) # 配列の一番奥の要素数を取る。
    scores = torch.matmul(Q, K.transpose(-1, -2))
    # Q(B, T_q, d_k)
    # K(B, T_k, d_k)^T = (B, d_k, T_k)
    # となり、QK^T(B, T_q, T_k)
    scores = scores / math.sqrt(d_k)
    # Attention weights
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf")) # Attentionのマスクにより見てはいけない場所の情報を消す。のちのsoftmaxで0
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V) # (B, T_q, d_V)
    return output, attention_weights


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0 #  "d_modelはnum_headsで割り切れること"
        self.d_model = d_model
        self.num_heads = num_heads # h
        self.d_k = d_model // num_heads

        # Q, K, Vを作る学習可能な線形変換
        self.W_q = nn.Linear(d_model, d_model)  # xW^Q + b
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        # 全headをconcatした後の線形変換
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """
        x: (B, T, d_model)
        return:
            (B, h, T, d_k)
        """
        B, T, _ = x.shape
        # (B, T, d_model) -> (B, T, h, d_k)
        x = x.view(B, T, self.num_heads, self.d_k)
        # -> (B, h, T, d_k)
        x = x.transpose(1, 2)
        return x

    def forward(self, Q, K, V, mask=None):
        B = Q.size(0)
        T_q = Q.size(1)
        # 線形射影
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)
        # Multi-headに分割
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        # Attention
        output, attention_weights = scaled_dot_product_attention(Q, K, V, mask) # ブロードキャスト
        # output:　(B, h, T_q, d_k)
        # headとsequenceの位置を入れ替える
        x = output.transpose(1, 2)
        # (B, T_q, h, d_k) -> (B, T_q, d_model)
        x = x.contiguous().view(B, T_q, self.d_model)
        # 最後のLinear
        output = self.W_o(x)
        return output, attention_weights

In [ ]:
# テスト
B = 2
T_q = 5
T_k = 7

d_model = 256
num_heads = 4 # h

Q = torch.randn(B, T_q, d_model)
K = torch.randn(B, T_k, d_model)
V = torch.randn(B, T_k, d_model)

mha = MultiHeadAttention(
    d_model=d_model,
    num_heads=num_heads
)

output, attention_weights = mha(Q, K, V)

print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

print("output:", output.shape)
print("attention_weights:", attention_weights.shape) # (B, h, Tq, Tk)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        # positional encoding
        pe = torch.zeros(max_len, d_model) # shape: (max_len, d_model)

        # position [0, 1, 2,...max_len-1] shape: (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # 周波数部分
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        p = position * div_term # (max_len, 1) * (d_model / 2, ) -> (max_len, d_model / 2)
        # 偶数次元
        pe[:, 0::2] = torch.sin(p)
        # 奇数次元
        pe[:, 1::2] = torch.cos(p)
        # batch次元を追加
        # (max_len, d_model) -> (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe) # GPUに移動しないためにregisterに格納

    def forward(self, x):
        # x: (B, T, d_model)
        T = x.size(1)
        x = x + self.pe[:, :T]
        return x


In [ ]:
# TEST
d_model = 8
max_len = 10
pos_encoding = PositionalEncoding(d_model=d_model, max_len=max_len)
print(pos_encoding.pe.shape)
print(pos_encoding.pe[0])

In [ ]:
# TEST 2
B = 2
T = 5
d_model = 8
x = torch.randn(B, T, d_model)
pos_encoding = PositionalEncoding(d_model=d_model)
output = pos_encoding(x)
print("input :", x.shape)
print("output:", output.shape)

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionwiseFeedForward, self).__init__()
        self.lenear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # (B, T, d_model)
        x = self.lenear1(x)
        # (B, T, d_ff)
        x = F.relu(x)
        # (B, T, d_model)
        x = self.linear2(x)
        return x

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.self_attention = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model=d_model, d_ff=d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Multi-head Self-Attention
        attention_outputs, attention_weights = self.self_attention(x, x, x, mask)
        # Add & Norm
        x = self.norm1(x + self.dropout(attention_outputs))
        # Feed Forward
        ff_output = self.feed_forward(x)
        # Add & Norm
        x = self.norm2(x + self.dropout(ff_output))
        return x, attention_weights

In [ ]:
#TEST
B = 2
T = 10
d_model = 256
num_heads = 4
d_ff = 1024
x = torch.randn(B, T, d_model)
encoder_layer = EncoderLayer(d_model=d_model, num_heads=num_heads, d_ff=d_ff)
output, attention_weights = encoder_layer(x)
print("input:")
print(x.shape)
print("output:")
print(output.shape)
print("attention:")
print(attention_weights.shape)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super(Encoder, self).__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model=d_model, num_heads=num_heads, d_ff=d_ff, dropout=dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        attention_weights = []
        for layer in self.layers:
            x, attention_weight = layer(x, mask)
            attention_weights.append(attention_weight)
        return x, attention_weights

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        # Masked Self-Attention
        self.self_attention = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
        # Encoder-Decoder Attention
        self.cross_attention = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
        # Feed Forward
        self.feed_forward = PositionwiseFeedForward(d_model=d_model, d_ff=d_ff)
        # Add & Norm
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Masked Self-Attention
        self_attention_outputs, self_attention_weights = self.self_attention(x, x, x, tgt_mask)
        # Add & Norm
        x = self.norm1(x + self.dropout(self_attention_outputs))
        cross_attention_outputs, cross_attention_weights = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attention_outputs))
        # Feed Forward
        ff_output = self.feed_forward(x)
        # Add & Norm
        x = self.norm3(x + self.dropout(ff_output))
        return x, self_attention_weights, cross_attention_weights

In [ ]:
class Decoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super(Decoder, self).__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        self_attention_weights = []
        cross_attention_weights = []
        for layer in self.layers:
            x, self_attention_weight, cross_attention_weight = layer(x, encoder_output, src_mask, tgt_mask)
            self_attention_weights.append(self_attention_weight)
            cross_attention_weights.append(cross_attention_weight)
        return x, self_attention_weights, cross_attention_weights

In [ ]:
# TEST
B = 2
T_src = 10
T_tgt = 7
d_model = 256
encoder_output = torch.randn(B, T_src, d_model)
decoder_input = torch.randn(B, T_tgt, d_model)
decoder = Decoder(num_layers=3, d_model=256, num_heads=4, d_ff=1024)
output, self_attns, cross_attns = decoder(decoder_input, encoder_output)
print("output:")
print(output.shape)
print("self attention:")
print(self_attns[0].shape)
print("cross attention:")
print(cross_attns[0].shape)

In [ ]:
def creating_padding_mask(seq, pad_id=0):
    """
    seq: (B, T)
    return: (B, 1, 1, T)
    """
    mask = (seq != pad_id)
    mask = mask.unsqueeze(1).unsqueeze(2)
    return mask

In [ ]:
def creating_causal_mask(T, device):
    """
    return:
        (1, 1, T, T)
    """
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))
    mask = mask.unsqueeze(0).unsqueeze(0)
    return mask

In [ ]:
def creating_decoder_mask(tgt, pad_id=0):
    # (B, 1, 1, T)
    padding_mask = creating_padding_mask(tgt, pad_id)
    # (1, 1, T, T)
    causal_mask = creating_causal_mask(tgt.size(1), tgt.device)
    # (B, 1, T, T)
    mask = torch.logical_and(padding_mask, causal_mask)
    return mask

In [ ]:
#TEST
tgt = torch.tensor([
    [2, 10, 20, 30, 3, 0],
    [2, 40, 50, 3, 0, 0]
])
tgt_mask = creating_decoder_mask(tgt)
print(tgt.shape)
print(tgt_mask.shape)
print(tgt_mask[0, 0].int())
print(tgt_mask[1, 0].int())

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, num_layers=3, num_heads=4, d_ff=1024, dropout=0.1, max_len=5000):
        super(Transformer, self).__init__()
        self.d_model = d_model
        # English Enbeddings
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        # Japanese Enbeddings
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        # Positional Encoding
        self.posional_encoding = PositionalEncoding(d_model=d_model, max_len=max_len)
        # Encoder
        self.encoder = Encoder(num_layers, d_model, num_heads, d_ff, dropout)
        # Decoder
        self.decoder = Decoder(num_layers, d_model, num_heads, d_ff, dropout)
        # d_model -> Japanese Words
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask, tgt_mask):
        # ========
        # Encoder
        # ========
        # token ID -> embedding
        src = self.src_embedding(src)
        # embedding scale
        src = src * math.sqrt(self.d_model)
        # positional encoding
        src = self.posional_encoding(src)
        # Encoder
        encoder_output, encoder_attentions = self.encoder(src, src_mask)
        # ========
        # Decoder
        # ========
        # token ID -> embedding
        tgt = self.tgt_embedding(tgt)
        # embedding scale
        tgt = tgt * math.sqrt(self.d_model)
        # positional_encoding
        tgt = self.posional_encoding(tgt)
        # Decoder
        decoder_output, decoder_self_attentions, decoder_cross_attentions = self.decoder(tgt, encoder_output, src_mask, tgt_mask)
        # ========
        # Output
        # ========
        logits = self.output_linear(decoder_output) # 学習時はSoftmaxはいれない。
        return (logits, encoder_attentions, decoder_self_attentions, decoder_cross_attentions)

In [ ]:
# TEST
src_vocab_size = 8000
tgt_vocab_size = 8000
model = Transformer(src_vocab_size=8000, tgt_vocab_size=8000, d_model=256, num_heads=4, num_layers=3, d_ff=1024, dropout=0.1)
B = 2
T_src = 10
T_tgt = 7
src = torch.randint(4, 8000, (B, T_src))
tgt = torch.randint(4, 8000, (B, T_tgt))
src_mask = creating_padding_mask(src)
tgt_mask = creating_decoder_mask(tgt)
logits, enc_attn, dec_self_attn, cross_attn = model(src, tgt, src_mask, tgt_mask)
print("logits:", logits.shape)
print("encoder attention:", enc_attn[0].shape)
print("decoder self attention:", dec_self_attn[0].shape)
print("cross attention:", cross_attn[0].shape)

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, dataset, sp_en, sp_ja):
        self.dataset = dataset
        self.sp_en = sp_en
        self.sp_ja = sp_ja

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]
        en = sample["translation"]["en"]
        ja = sample["translation"]["ja"]
        # English
        src_ids = self.sp_en.encode(en, out_type=int, add_bos=True, add_eos=True)
        # Japanese
        tgt_ids = self.sp_ja.encode(ja, out_type=int, add_bos=True, add_eos=True)

        src = torch.tensor(src_ids, dtype=torch.long)
        tgt = torch.tensor(tgt_ids, dtype=torch.long)
        return src, tgt


In [ ]:
from torch.nn.utils.rnn import pad_sequence

PAD_ID = 0

def collate_fn(batch):
    src_batch = []
    tgt_batch = []
    for src, tgt in batch:
        src_batch.append(src)
        tgt_batch.append(tgt)

    # (B, max_src_len)
    src_batch = pad_sequence(src_batch, padding_value=PAD_ID, batch_first=True)
    # (B, max_tgt_len)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_ID, batch_first=True)
    return src_batch, tgt_batch

In [ ]:
train_data = TranslationDataset(train_dataset, sp_en, sp_ja)
train_loader = DataLoader(train_data, batch_size=320, shuffle=True, collate_fn=collate_fn)
val_data = TranslationDataset(val_dataset, sp_en, sp_ja)
val_loader = DataLoader(val_data, batch_size=320, shuffle=False, collate_fn=collate_fn)

In [ ]:
# TEST
src, tgt = next(iter(train_loader))

print(src.shape)
print(tgt.shape)

In [ ]:
#TEST
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(src_vocab_size=len(sp_en), tgt_vocab_size=len(sp_ja), d_model=256, num_heads=4, num_layers=3, d_ff=1024, dropout=0.1).to(device)
src, tgt = next(iter(train_loader))
src = src.to(device)
tgt = tgt.to(device)
# Teacher Forcing
decoder_input = tgt[:, :-1]
target = tgt[:, 1:]
src_mask = creating_padding_mask(src, PAD_ID)
tgt_mask = creating_decoder_mask(decoder_input, PAD_ID)
logits, enc_attn, dec_self_attn, dec_cross_attn = model(src, decoder_input, src_mask, tgt_mask)
print("src:", src.shape)
print("tgt:", tgt.shape)
print("decoder_input:", decoder_input.shape)
print("target:", target.shape)
print("logits:", logits.shape)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
loss = criterion(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
print("loss:", loss.item())

In [ ]:
class TransformerLRScheduler:
    def __init__(self, optimizer, d_model, warmup_steps=4000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def step(self):
        self.step_num += 1
        lr = self.d_model ** (-0.5) * min(self.step_num ** (-0.5), self.step_num * self.warmup_steps ** (-1.5))
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr
        return lr

In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for src, tgt in data_loader:
            src = src.to(device)
            tgt = tgt.to(device)
            decoder_input = tgt[:, :-1]
            target = tgt[:, 1:]
            src_mask = creating_padding_mask(src, PAD_ID)
            tgt_mask = creating_decoder_mask(decoder_input, PAD_ID)
            logits, _, _, _ = model(src, decoder_input, src_mask, tgt_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(data_loader)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(src_vocab_size=len(sp_en), tgt_vocab_size=len(sp_ja), d_model=256, num_heads=4, num_layers=3, d_ff=1024).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = TransformerLRScheduler(optimizer, d_model=256, warmup_steps=4000)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)
start_epoch = 0
train_losses = []
val_losses = []

CHECKPOINT_DIR = "/content/gdrive/MyDrive/深層学習スクラッチ/deeplearning_implementation/Transformer/transformer_jesc/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint.pt")
if os.path.exists(CHECKPOINT_PATH):
    print("Checkpoint found. Loading...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    # Model
    model.load_state_dict(checkpoint["model_state_dict"])
    # Adam
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    # Scheduler
    scheduler.step_num = checkpoint["scheduler_step"]
    # 次に開始するepoch
    start_epoch = checkpoint["epoch"] + 1
    # Loss履歴
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    print(f"Resuming from epoch {start_epoch + 1}")
else:
    print("No checkpoint found.")
    print("Starting training from scratch.")

    num_epochs = 100
    for epoch in range(start_epoch, num_epochs):
        model.train()
        total_loss = 0.0
        for batch_idx, (src, tgt) in enumerate(train_loader):
            src = src.to(device)
            tgt = tgt.to(device)
            # ==========================
            # Teacher Forcing
            # ==========================
            decoder_input = tgt[:, :-1]
            target = tgt[:, 1:]
            # ==========================
            # Mask
            # ==========================
            src_mask = creating_padding_mask(src, PAD_ID)
            tgt_mask = creating_decoder_mask(decoder_input, PAD_ID)
            # ==========================
            # Forward
            # ==========================
            logits, _, _, _ = model(src, decoder_input, src_mask, tgt_mask)
            # ==========================
            # Loss
            # ==========================
            loss = criterion(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
            # ==========================
            # Backward
            # ==========================
            optimizer.zero_grad()
            loss.backward()
            # ==========================
            # Learning Rate Update
            # ==========================
            lr = scheduler.step()
            # ==========================
            # Parameter Update
            # ==========================
            optimizer.step()


            total_loss += loss.item()
            if (batch_idx + 1) % 100 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}, LR {lr:.8f}")


        train_loss = total_loss / len(train_loader)
        val_loss = evaluate(model, val_loader, criterion, device)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epoch {epoch+1} " f"Train Loss: {train_loss:.4f}" f"Val Loss = {val_loss:.4f}\n")

        # ==========================
        # Checkpoint Save
        # ==========================
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_step": scheduler.step_num,
                "train_losses": train_losses,
                "val_losses": val_losses,
            },
            CHECKPOINT_PATH
        )
        print(f"Checkpoint saved: "f"epoch {epoch+1}")

In [ ]:
import matplotlib.pyplot as plt
epochs = range(1, num_epochs + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, marker="o", label="Train Loss")
plt.plot(epochs, val_losses, marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
def translate(model, text, sp_en, sp_ja, device, max_len=50):
    model.eval()
    # =========================
    # English tokenize
    # =========================
    src_ids = sp_en.encode(text, out_type=int, add_bos=True, add_eos=True)
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    # =========================
    # Source mask
    # =========================
    src_mask = creating_padding_mask(src, sp_en.pad_id())
    # =========================
    # <bos>から開始
    # =========================
    BOS_ID = sp_ja.bos_id()
    EOS_ID = sp_ja.eos_id()
    PAD_ID = sp_ja.pad_id()
    generated = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)
    # =========================
    # Autoregressive generation
    # =========================
    with torch.no_grad():
        for _ in range(max_len):
            tgt_mask = creating_decoder_mask(generated, PAD_ID)
            logits, _, _, _ = model(src, generated, src_mask, tgt_mask)
            # 最後のtoken位置だけ取得
            next_token_logits = logits[:, -1, :]
            # Greedy
            next_token = torch.argmax(next_token_logits, dim=-1)
            # (1,) -> (1,1)
            next_token = next_token.unsqueeze(1)
            # generatedに追加
            generated = torch.cat([generated, next_token], dim=1)
            # <eos>なら終了
            if next_token.item() == EOS_ID:
                break

    # =========================
    # token IDs -> Japanese
    # =========================
    generated_ids = generated[0].tolist()
    translation = sp_ja.decode(generated_ids)
    return translation

In [ ]:
result = translate(model, "I like cats.", sp_en, sp_ja, device)
print("TEST")
print(result)

In [ ]:
result = translate(model, "I am playing a guitar.", sp_en, sp_ja, device)
print("TEST")
print(result)